In [ ]:
import snowflake.connector
import pandas as pd
import os

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA")
)

df = pd.read_sql("SELECT * FROM DIABETES_TABLE", conn)
df

/tmp/ipykernel_300/1136124994.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM DIABETES_TABLE", conn)


,ID,NO_PATION,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,200,454317,M,71,11.0,97,7.0,7.5,1.7,1.2,1.8,0.6,30.0,Y
996,671,876534,M,31,3.0,60,12.3,4.1,2.2,0.7,2.4,15.4,37.2,Y
997,669,87654,M,30,7.1,81,6.7,4.1,1.1,1.2,2.4,8.1,27.4,Y
998,99,24004,M,38,5.8,59,6.7,5.3,2.0,1.6,2.9,14.0,40.5,Y


In [2]:
import matplotlib.pyplot as plt
import numpy as np
!pip install imbalanced-learn 

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 4.8 MB/s eta 0:00:00a 0:00:01


In [3]:
diabetes = df.copy()
diabetes

,ID,NO_PATION,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,200,454317,M,71,11.0,97,7.0,7.5,1.7,1.2,1.8,0.6,30.0,Y
996,671,876534,M,31,3.0,60,12.3,4.1,2.2,0.7,2.4,15.4,37.2,Y
997,669,87654,M,30,7.1,81,6.7,4.1,1.1,1.2,2.4,8.1,27.4,Y
998,99,24004,M,38,5.8,59,6.7,5.3,2.0,1.6,2.9,14.0,40.5,Y


EDA

In [4]:
diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1000 non-null   int64  
 1   NO_PATION  1000 non-null   int64  
 2   GENDER     1000 non-null   object 
 3   AGE        1000 non-null   int64  
 4   UREA       1000 non-null   float64
 5   CR         1000 non-null   int64  
 6   HBA1C      1000 non-null   float64
 7   CHOL       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  CLASE      1000 non-null   object 
dtypes: float64(8), int64(4), object(2)
memory usage: 109.5+ KB


In [5]:
diabetes.describe()

,ID,NO_PATION,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,340.500000,2.705514e+05,53.528000,5.124743,68.943000,8.281160,4.862820,2.349610,1.204750,2.609790,1.854700,29.578020
std,240.397673,3.380758e+06,8.799241,2.935165,59.984747,2.534003,1.301738,1.401176,0.660414,1.115102,3.663599,4.962388
min,1.000000,1.230000e+02,20.000000,0.500000,6.000000,0.900000,0.000000,0.300000,0.200000,0.300000,0.100000,19.000000
25%,125.750000,2.406375e+04,51.000000,3.700000,48.000000,6.500000,4.000000,1.500000,0.900000,1.800000,0.700000,26.000000
50%,300.500000,3.439550e+04,55.000000,4.600000,60.000000,8.000000,4.800000,2.000000,1.100000,2.500000,0.900000,30.000000
75%,550.250000,4.538425e+04,59.000000,5.700000,73.000000,10.200000,5.600000,2.900000,1.300000,3.300000,1.500000,33.000000
max,800.000000,7.543566e+07,79.000000,38.900000,800.000000,16.000000,10.300000,13.800000,9.900000,9.900000,35.000000,47.750000


In [6]:
diabetes.isnull().sum()

ID           0
NO_PATION    0
GENDER       0
AGE          0
UREA         0
CR           0
HBA1C        0
CHOL         0
TG           0
HDL          0
LDL          0
VLDL         0
BMI          0
CLASE        0
dtype: int64

In [7]:
encoder = LabelEncoder()
diabetes['GENDER'] = encoder.fit_transform(diabetes['GENDER'])
diabetes.head()

,ID,NO_PATION,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,502,17975,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,1,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,1,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [8]:
diabetes.drop(['NO_PATION', 'ID'], axis=1, inplace=True)
diabetes.head()

,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,1,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,1,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [9]:
diabetes['CLASE'] = diabetes['CLASE'].str.strip()
print(diabetes['CLASE'].unique())

['N' 'P' 'Y']


In [10]:
diabetes.shape

(1000, 12)

In [11]:
X = diabetes.drop('CLASE', axis=1)
y = diabetes['CLASE']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Standard Scaler

In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
X_train.shape, y_train.shape

((700, 11), (700,))

In [15]:
X_test.shape, y_test.shape

((300, 11), (300,))

In [16]:
X_test_scaled.shape, y_test.shape

((300, 11), (300,))

Modeling

KNN

In [17]:
knn = KNeighborsClassifier(n_neighbors=49)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print("KNN:")
print(classification_report(y_test, y_pred_knn, zero_division=0))

KNN:
              precision    recall  f1-score   support

           N       0.70      0.44      0.54        36
           P       0.00      0.00      0.00        10
           Y       0.91      0.99      0.95       254

    accuracy                           0.89       300
   macro avg       0.53      0.48      0.50       300
weighted avg       0.85      0.89      0.87       300



KNN with different parameter

In [18]:
knn_new = KNeighborsClassifier(n_neighbors=7)
knn_new.fit(X_train, y_train)
y_pred_knn_new = knn_new.predict(X_test)

print("KNN new:")
print(classification_report(y_test, y_pred_knn_new, zero_division=0))

KNN new:
              precision    recall  f1-score   support

           N       0.75      0.67      0.71        36
           P       0.25      0.40      0.31        10
           Y       0.96      0.96      0.96       254

    accuracy                           0.90       300
   macro avg       0.65      0.67      0.66       300
weighted avg       0.91      0.90      0.91       300



KNN with data Scaled

In [19]:
knn_new_s = KNeighborsClassifier(n_neighbors=7)
knn_new_s.fit(X_train_scaled, y_train)
y_pred_knn_new_s = knn_new_s.predict(X_test_scaled)

print("KNN new scaled:")
print(classification_report(y_test, y_pred_knn_new_s, zero_division=0))

KNN new scaled:
              precision    recall  f1-score   support

           N       0.84      0.89      0.86        36
           P       0.75      0.30      0.43        10
           Y       0.97      0.98      0.97       254

    accuracy                           0.95       300
   macro avg       0.85      0.72      0.76       300
weighted avg       0.94      0.95      0.94       300



Naive Bayes

In [20]:
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Naive Bayes:")
print(classification_report(y_test, y_pred_nb, zero_division=0))

Naive Bayes:
              precision    recall  f1-score   support

           N       0.74      0.89      0.81        36
           P       0.82      0.90      0.86        10
           Y       0.98      0.95      0.96       254

    accuracy                           0.94       300
   macro avg       0.85      0.91      0.88       300
weighted avg       0.95      0.94      0.94       300



Support Vector Machine (SVM)

In [21]:
svm = SVC(kernel='linear', C=1.0)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("Support Vector Machine:")
print(classification_report(y_test, y_pred_svm, zero_division=0))

Support Vector Machine:
              precision    recall  f1-score   support

           N       0.78      0.89      0.83        36
           P       0.60      0.30      0.40        10
           Y       0.96      0.96      0.96       254

    accuracy                           0.93       300
   macro avg       0.78      0.72      0.73       300
weighted avg       0.93      0.93      0.93       300



Decision Tree

In [22]:
dt = DecisionTreeClassifier(max_depth=2)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree:")
print(classification_report(y_test, y_pred_dt, zero_division=0))

Decision Tree:
              precision    recall  f1-score   support

           N       0.73      1.00      0.85        36
           P       0.43      1.00      0.61        10
           Y       1.00      0.90      0.95       254

    accuracy                           0.91       300
   macro avg       0.72      0.97      0.80       300
weighted avg       0.95      0.91      0.92       300



Random Forest

In [23]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest:
              precision    recall  f1-score   support

           N       0.97      0.92      0.94        36
           P       1.00      0.90      0.95        10
           Y       0.98      1.00      0.99       254

    accuracy                           0.98       300
   macro avg       0.99      0.94      0.96       300
weighted avg       0.98      0.98      0.98       300



Logistic Regression-in

In [24]:
log_reg = LogisticRegression(solver='saga')

log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

print("Logistic Regression:")
print(classification_report(y_test, y_pred_log_reg, zero_division=0))

Logistic Regression:
              precision    recall  f1-score   support

           N       0.00      0.00      0.00        36
           P       0.00      0.00      0.00        10
           Y       0.85      1.00      0.92       254

    accuracy                           0.85       300
   macro avg       0.28      0.33      0.31       300
weighted avg       0.72      0.85      0.78       300



/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression without Scaled 

In [25]:
log_no_scaler = LogisticRegression(solver='saga', max_iter=1000)
log_no_scaler.fit(X_train, y_train)

/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'saga'
,max_iter,1000
,multi_class,'deprecated'


TABELA 1 — Models without Scaled

In [26]:
from sklearn.metrics import precision_recall_fscore_support

models_no_scaler = {
    "KNN (n=7, sem scaler)": knn_new,
    "Naive Bayes (sem scaler)": nb,
    "SVM (sem scaler)": svm,
    "Decision Tree (sem scaler)": dt,
    "Random Forest (sem scaler)": rf,
    "Logistic Regression (sem scaler)": log_no_scaler
}

results_no_scaler = []

for name, model in models_no_scaler.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average="weighted"
    )
    results_no_scaler.append([name, acc, precision, recall, f1])

df_no_scaler = pd.DataFrame(
    results_no_scaler,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

df_no_scaler


/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Model,Accuracy,Precision,Recall,F1
0,"KNN (n=7, sem scaler)",0.903333,0.914762,0.903333,0.908164
1,Naive Bayes (sem scaler),0.940000,0.946033,0.940000,0.941973
2,SVM (sem scaler),0.930000,0.926992,0.930000,0.926407
3,Decision Tree (sem scaler),0.913333,0.949323,0.913333,0.922845
4,Random Forest (sem scaler),0.983333,0.983293,0.983333,0.983104
5,Logistic Regression (sem scaler),0.846667,0.780811,0.846667,0.787855


Training models with Scaler

In [27]:
knn_scaled_final = KNeighborsClassifier(n_neighbors=7)
knn_scaled_final.fit(X_train_scaled, y_train)

nb_scaled_final = GaussianNB()
nb_scaled_final.fit(X_train_scaled, y_train)

svm_scaled_final = SVC(kernel='linear', C=1.0)
svm_scaled_final.fit(X_train_scaled, y_train)

dt_scaled_final = DecisionTreeClassifier(max_depth=2)
dt_scaled_final.fit(X_train_scaled, y_train)

rf_scaled_final = RandomForestClassifier(n_estimators=100)
rf_scaled_final.fit(X_train_scaled, y_train)

lr_scaled_final = LogisticRegression(solver='saga', max_iter=1000)
lr_scaled_final.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'saga'
,max_iter,1000
,multi_class,'deprecated'


TABELA 2 — Models with Scaler

In [28]:
models_scaled = {
    "KNN (scaled)": knn_scaled_final,
    "Naive Bayes (scaled)": nb_scaled_final,
    "SVM (scaled)": svm_scaled_final,
    "Decision Tree (scaled)": dt_scaled_final,
    "Random Forest (scaled)": rf_scaled_final,
    "Logistic Regression (scaled)": lr_scaled_final
}

results_scaled = []

for name, model in models_scaled.items():
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average="weighted"
    )
    results_scaled.append([name, acc, precision, recall, f1])

df_scaled = pd.DataFrame(
    results_scaled,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

df_scaled

,Model,Accuracy,Precision,Recall,F1
0,KNN (scaled),0.946667,0.943184,0.946667,0.941585
1,Naive Bayes (scaled),0.940000,0.946033,0.940000,0.941973
2,SVM (scaled),0.943333,0.949302,0.943333,0.939002
3,Decision Tree (scaled),0.913333,0.949323,0.913333,0.922845
4,Random Forest (scaled),0.983333,0.983489,0.983333,0.982500
5,Logistic Regression (scaled),0.943333,0.938239,0.943333,0.936280


Top 3 Worst Results:

KNN x Decision Tree x Logistic Regression-in - Adjusting Imbalanced Classes

In [29]:
print("Distribuição antes do SMOTE:", y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote_raw, y_train_smote_raw = smote.fit_resample(X_train, y_train)

print("SMOTE sem scaler:", y_train_smote_raw.value_counts())

smote = SMOTE(random_state=42)
X_train_smote_scaled, y_train_smote_scaled = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE com scaler:", y_train_smote_scaled.value_counts())

Distribuição antes do SMOTE: CLASE
Y    590
N     67
P     43
Name: count, dtype: int64
SMOTE sem scaler: CLASE
Y    590
N    590
P    590
Name: count, dtype: int64
SMOTE com scaler: CLASE
Y    590
N    590
P    590
Name: count, dtype: int64


KNN with SMOTE

In [30]:
knn_raw = KNeighborsClassifier(n_neighbors=7)
knn_raw.fit(X_train_smote_raw, y_train_smote_raw)

y_pred_knn_raw = knn_raw.predict(X_test)

print("\nKNN — SMOTE sem scaler")
print(classification_report(y_test, y_pred_knn_raw, zero_division=0))


KNN — SMOTE sem scaler
              precision    recall  f1-score   support

           N       0.57      0.69      0.62        36
           P       0.21      0.60      0.32        10
           Y       1.00      0.90      0.95       254

    accuracy                           0.86       300
   macro avg       0.59      0.73      0.63       300
weighted avg       0.92      0.86      0.89       300



KNN with SMOTE and scaler

In [31]:
knn_scaled = KNeighborsClassifier(n_neighbors=7)
knn_scaled.fit(X_train_smote_scaled, y_train_smote_scaled)

y_pred_knn_scaled = knn_scaled.predict(X_test_scaled)

print("\nKNN — SMOTE com scaler")
print(classification_report(y_test, y_pred_knn_scaled, zero_division=0))


KNN — SMOTE com scaler
              precision    recall  f1-score   support

           N       0.70      0.83      0.76        36
           P       0.28      0.70      0.40        10
           Y       1.00      0.91      0.95       254

    accuracy                           0.89       300
   macro avg       0.66      0.81      0.70       300
weighted avg       0.94      0.89      0.91       300



Decision Tree with SMOTE

In [32]:
dt_raw = DecisionTreeClassifier(max_depth=2)
dt_raw.fit(X_train_smote_raw, y_train_smote_raw)
y_pred_dt_raw = dt_raw.predict(X_test)

print("\nDecision Tree — SMOTE sem scaler")
print(classification_report(y_test, y_pred_dt_raw, zero_division=0))


Decision Tree — SMOTE sem scaler
              precision    recall  f1-score   support

           N       0.86      1.00      0.92        36
           P       0.43      1.00      0.61        10
           Y       1.00      0.93      0.96       254

    accuracy                           0.94       300
   macro avg       0.76      0.98      0.83       300
weighted avg       0.96      0.94      0.94       300



Decision Tree with SMOTE and scaler

In [33]:
dt_scaled = DecisionTreeClassifier(max_depth=2)
dt_scaled.fit(X_train_smote_scaled, y_train_smote_scaled)
y_pred_dt_scaled = dt_scaled.predict(X_test_scaled)

print("\nDecision Tree — SMOTE com scaler")
print(classification_report(y_test, y_pred_dt_scaled, zero_division=0))


Decision Tree — SMOTE com scaler
              precision    recall  f1-score   support

           N       0.86      1.00      0.92        36
           P       0.43      1.00      0.61        10
           Y       1.00      0.93      0.96       254

    accuracy                           0.94       300
   macro avg       0.76      0.98      0.83       300
weighted avg       0.96      0.94      0.94       300



Logistic Regression-in with SMOTE

In [34]:
lr_raw = LogisticRegression(solver='saga', max_iter=1000)
lr_raw.fit(X_train_smote_raw, y_train_smote_raw)
y_pred_lr_raw = lr_raw.predict(X_test)

print("\nLogistic Regression — SMOTE sem scaler")
print(classification_report(y_test, y_pred_lr_raw, zero_division=0))


Logistic Regression — SMOTE sem scaler
              precision    recall  f1-score   support

           N       0.42      0.72      0.53        36
           P       0.10      0.50      0.16        10
           Y       0.98      0.72      0.83       254

    accuracy                           0.71       300
   macro avg       0.50      0.65      0.51       300
weighted avg       0.88      0.71      0.77       300



/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Logistic Regression with SMOTE and scaler

In [35]:
lr_scaled = LogisticRegression(solver='saga', max_iter=1000)
lr_scaled.fit(X_train_smote_scaled, y_train_smote_scaled)
y_pred_lr_scaled = lr_scaled.predict(X_test_scaled)

print("\nLogistic Regression — SMOTE com scaler")
print(classification_report(y_test, y_pred_lr_scaled, zero_division=0))


Logistic Regression — SMOTE com scaler
              precision    recall  f1-score   support

           N       0.75      0.92      0.82        36
           P       0.39      0.70      0.50        10
           Y       1.00      0.93      0.96       254

    accuracy                           0.92       300
   macro avg       0.71      0.85      0.76       300
weighted avg       0.95      0.92      0.93       300



Grid Search x Random Search - Random Forest 

Grid Search

In [36]:
param_grid = {
    "n_estimators": [50, 75, 100],
    "max_depth": [None, 5, 10, 20],
    "class_weight": [None, "balanced"]
}

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="accuracy",     
    cv=5,
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

print("Melhores hiperparâmetros:", grid_search_rf.best_params_)
print("Cross Validation:", grid_search_rf.best_score_)

best_rf_grid = grid_search_rf.best_estimator_
y_pred_rf_grid = best_rf_grid.predict(X_test)

print("\nRandom Forest (GridSearch)")
print(classification_report(y_test, y_pred_rf_grid, zero_division=0))

Melhores hiperparâmetros: {'class_weight': None, 'max_depth': None, 'n_estimators': 75}
Cross Validation: 0.9814285714285715

Random Forest (GridSearch)
              precision    recall  f1-score   support

           N       0.97      0.97      0.97        36
           P       1.00      0.70      0.82        10
           Y       0.98      1.00      0.99       254

    accuracy                           0.98       300
   macro avg       0.99      0.89      0.93       300
weighted avg       0.98      0.98      0.98       300



Random Search

In [37]:
rf2 = RandomForestClassifier(random_state=42)

param_dist = {
    "n_estimators": np.arange(50, 75, 100),  
    "max_depth": [None, 5, 10, 20, 30],
    "class_weight": [None, "balanced"]
}

rand_search_rf = RandomizedSearchCV(
    estimator=rf2,
    param_distributions=param_dist,
    n_iter=20,            
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rand_search_rf.fit(X_train, y_train)

print("Melhores hiperparâmetros:", rand_search_rf.best_params_)
print("Cross Validation:", rand_search_rf.best_score_)

best_rf_rand = rand_search_rf.best_estimator_
y_pred_rf_rand = best_rf_rand.predict(X_test)

print("\nRandom Forest (RandomizedSearch) - Teste:")
print(classification_report(y_test, y_pred_rf_rand, zero_division=0))


/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 10 is smaller than n_iter=20. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Melhores hiperparâmetros: {'n_estimators': np.int64(50), 'max_depth': None, 'class_weight': None}
Cross Validation: 0.9842857142857143

Random Forest (RandomizedSearch) - Teste:
              precision    recall  f1-score   support

           N       0.97      0.94      0.96        36
           P       1.00      0.90      0.95        10
           Y       0.99      1.00      0.99       254

    accuracy                           0.99       300
   macro avg       0.99      0.95      0.97       300
weighted avg       0.99      0.99      0.99       300



In [38]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score
from mlflow.models.signature import infer_signature

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("predicting-diabetes")

def log_model_mlflow(model, X_test, y_test, artifact_name: str):
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    signature = infer_signature(X_test, preds)
    input_example = X_test.iloc[:1]

    with mlflow.start_run():
        mlflow.log_params(model.get_params())
        mlflow.log_metric("accuracy", acc)
        
        mlflow.sklearn.log_model(
            model,
            name=artifact_name,
            signature=signature,
            input_example=input_example
        )

    print(f"Modelo '{artifact_name}' salvo no MLflow. Accuracy={acc:.4f}")

models = {
    "knn": knn_scaled_final,
    "naive_bayes": nb_scaled_final,
    "svm": svm_scaled_final,
    "decision_tree": dt_scaled_final,
    "random_forest": best_rf_grid,
    "log_reg": lr_scaled_final
}

for name, model in models.items():
    log_model_mlflow(model, X_test, y_test, name)


2025/12/02 23:53:51 INFO mlflow.tracking.fluent: Experiment with name 'predicting-diabetes' does not exist. Creating a new experiment.
/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handli

🏃 View run amazing-stag-809 at: http://mlflow:5000/#/experiments/1/runs/d804b7d282784e23a09bea5e56b6dcd6
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Modelo 'knn' salvo no MLflow. Accuracy=0.8467


/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but GaussianNB was fitted without feature names
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: U

🏃 View run casual-fly-311 at: http://mlflow:5000/#/experiments/1/runs/0973d179e5594db2adff6c2f4f8d5165
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Modelo 'naive_bayes' salvo no MLflow. Accuracy=0.8467


/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but SVC was fitted without feature names
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarn

...

🏃 View run auspicious-lynx-141 at: http://mlflow:5000/#/experiments/1/runs/cb1c81fb204b45aab9cee04ed7e70b5f
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Modelo 'decision_tree' salvo no MLflow. Accuracy=0.8467


/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run polite-ray-914 at: http://mlflow:5000/#/experiments/1/runs/ebf77579c8fe4649a931c23cabe430d0
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Modelo 'random_forest' salvo no MLflow. Accuracy=0.9833


/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py

🏃 View run mercurial-koi-237 at: http://mlflow:5000/#/experiments/1/runs/4f0b016022574f19966dba250b8d1830
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Modelo 'log_reg' salvo no MLflow. Accuracy=0.8467


In [ ]:
import pandas as pd
import requests
import time

# ================= CONFIGURAÇÕES =================
# URL do ThingsBoard visto do HOST (se você acessa no navegador via localhost:9090, é isso aqui mesmo)
TB_BASE_URL = "http://127.0.0.1:9090"

# Token de acesso do *device* que você criou no ThingsBoard
DEVICE_TOKEN = "L5Aisizs46cSREYpQTeM"  # <-- TROCAR PELO SEU

# Caminho do CSV no seu Windows
CSV_PATH = r"C:\Users\gabri\OneDrive\Desktop\ML\Predicting-Diabetes\data\raw\Dataset_of_Diabetes.csv"

# Atraso entre requisições (segundos) só pra não entupir o TB
SLEEP_SECONDS = 0.01
# =================================================


def main():
    print(f"Lendo CSV: {CSV_PATH}")
    df = pd.read_csv(CSV_PATH)
    print(f"Total de linhas: {len(df)}")

    url = f"{TB_BASE_URL}/api/v1/{DEVICE_TOKEN}/telemetry"
    headers = {"Content-Type": "application/json"}

    total = len(df)

    for idx, (_, row) in enumerate(df.iterrows(), start=1):
        payload = row.to_dict()

        # Limpeza básica pra garantir que tudo é serializável em JSON
        clean_payload = {}
        for k, v in payload.items():
            if pd.isna(v):
                clean_payload[k] = None
            else:
                # Converte tipos numpy para tipos Python "normais"
                try:
                    clean_payload[k] = v.item() if hasattr(v, "item") else v
                except Exception:
                    clean_payload[k] = str(v)

        try:
            resp = requests.post(url, json=clean_payload, headers=headers, timeout=30)
        except requests.RequestException as e:
            print(f"[{idx}/{total}] ERRO de conexão: {e}")
            break

        if resp.status_code != 200:
            print(f"[{idx}/{total}] ERRO HTTP {resp.status_code}: {resp.text}")
            break


        if resp.status_code != 200:
            print(f"[{idx}/{total}] ERRO HTTP {resp.status_code}: {resp.text}")
            break

        if idx % 100 == 0 or idx == total:
            print(f"Enviadas {idx}/{total} linhas")

        time.sleep(SLEEP_SECONDS)

    print("Finalizado.")


if __name__ == "__main__":
    main()


In [6]:
import pandas as pd
import requests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


CSV_PATH = r"C:\Users\gabri\OneDrive\Desktop\ML\Predicting-Diabetes\data\raw\Dataset_of_Diabetes.csv"
DEVICE_TOKEN = "L5Aisizs46cSREYpQTeM"
TB_BASE_URL = "http://127.0.0.1:9090"

url = f"{TB_BASE_URL}/api/v1/{DEVICE_TOKEN}/telemetry"

diabetes = pd.read_csv(CSV_PATH)

# Limpar espaços
diabetes['CLASS'] = diabetes['CLASS'].str.strip()
diabetes['Gender'] = diabetes['Gender'].str.strip()

# Transformar Gender em número (exemplo: M=0, F=1)
diabetes['Gender'] = diabetes['Gender'].map({'M': 0, 'F': 1})

print("Valores únicos de CLASS:", diabetes['CLASS'].unique())
print("Valores únicos de Gender:", diabetes['Gender'].unique())

X = diabetes.drop('CLASS', axis=1)
y = diabetes['CLASS']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Garante que as chaves N, P, Y existam no dict
labels = ["N", "P", "Y"]

report = classification_report(
    y_test,
    y_pred_rf,
    labels=labels,
    target_names=labels,
    output_dict=True,
    zero_division=0,
)
accuracy = accuracy_score(y_test, y_pred_rf)

print("Random Forest:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

metrics_payload = {
    # Métrica geral
    "rf_accuracy": accuracy,

    # Classe N
    "rf_precision_N": report["N"]["precision"],
    "rf_recall_N": report["N"]["recall"],
    "rf_f1_N": report["N"]["f1-score"],

    # Classe P
    "rf_precision_P": report["P"]["precision"],
    "rf_recall_P": report["P"]["recall"],
    "rf_f1_P": report["P"]["f1-score"],

    # Classe Y
    "rf_precision_Y": report["Y"]["precision"],
    "rf_recall_Y": report["Y"]["recall"],
    "rf_f1_Y": report["Y"]["f1-score"],

    # Macro avg
    "rf_precision_macro": report["macro avg"]["precision"],
    "rf_recall_macro": report["macro avg"]["recall"],
    "rf_f1_macro": report["macro avg"]["f1-score"],
}

response = requests.post(url, json=metrics_payload, timeout=5)
print("Status:", response.status_code, response.text)


Valores únicos de CLASS: ['N' 'P' 'Y']
Valores únicos de Gender: [ 1.  0. nan]
Random Forest:
              precision    recall  f1-score   support

           N       0.97      0.92      0.94        36
           P       1.00      0.80      0.89        10
           Y       0.98      1.00      0.99       254

    accuracy                           0.98       300
   macro avg       0.98      0.90      0.94       300
weighted avg       0.98      0.98      0.98       300

Status: 200 


In [ ]:
import pandas as pd
import requests
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier


CSV_PATH = r"C:\Users\gabri\OneDrive\Desktop\ML\Predicting-Diabetes\data\raw\Dataset_of_Diabetes.csv"
DEVICE_TOKEN = "L5Aisizs46cSREYpQTeM"
TB_BASE_URL = "http://127.0.0.1:9090"



url = f"{TB_BASE_URL}/api/v1/{DEVICE_TOKEN}/telemetry"

diabetes = pd.read_csv(CSV_PATH)

diabetes['CLASS'] = diabetes['CLASS'].str.strip()
print(diabetes['CLASS'].unique())

# Transformar Gender em número (exemplo: M=0, F=1)
diabetes['Gender'] = diabetes['Gender'].map({'M': 0, 'F': 1})

X = diabetes.drop('CLASS', axis=1)
y = diabetes['CLASS']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

dt_report = classification_report(
    y_test, y_pred_dt, output_dict=True, zero_division=0
)
dt_accuracy = accuracy_score(y_test, y_pred_dt)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

dt_sm = DecisionTreeClassifier(random_state=42)
dt_sm.fit(X_train_sm, y_train_sm)
y_pred_dt_sm = dt_sm.predict(X_test)

dt_sm_report = classification_report(
    y_test, y_pred_dt_sm, output_dict=True, zero_division=0
)
dt_sm_accuracy = accuracy_score(y_test, y_pred_dt_sm)

metrics_payload = {

    # -------------------------
    # Decision Tree (sem SMOTE)
    # -------------------------
    "dt_accuracy": dt_accuracy,
    "dt_f1_N": dt_report["N"]["f1-score"],
    "dt_f1_P": dt_report["P"]["f1-score"],
    "dt_f1_Y": dt_report["Y"]["f1-score"],
    "dt_f1_macro": dt_report["macro avg"]["f1-score"],

    # -------------------------
    # Decision Tree (com SMOTE)
    # -------------------------
    "dt_smote_accuracy": dt_sm_accuracy,
    "dt_smote_f1_N": dt_sm_report["N"]["f1-score"],
    "dt_smote_f1_P": dt_sm_report["P"]["f1-score"],
    "dt_smote_f1_Y": dt_sm_report["Y"]["f1-score"],
    "dt_smote_f1_macro": dt_sm_report["macro avg"]["f1-score"],
}

response = requests.post(url, json=metrics_payload, timeout=5)
print("Status:", response.status_code, response.text)

['N' 'P' 'Y']


ValueError: could not convert string to float: 'M'